[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/2_Practica/para_casa/S06_02_limpieza.ipynb)

# S06 · 02 — Limpieza de datos: encontrar defectos y sus causas

Sprint 6 · Módulo M2 (UD 2.4)

**Contexto TurisData.** El cliente nos envía un segundo fichero, `reservas_sucio.csv`, exportado "a mano" desde varios sistemas. Antes de usarlo hay que **localizar los defectos, entender de dónde vienen y corregirlos con reglas claras**. Limpiar no es "borrar lo feo": cada decisión (eliminar, corregir, dejar a nulo) debe estar justificada.

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 1, 2, 3 y 5 son los mismos que los del cuaderno «en clase» (allí, del 1 al 4).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
sns.set_theme(style="whitegrid")
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar el fichero sucio

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas filas y cuántas columnas tiene `crudo`?
2. ¿Qué columnas están como texto (`object`) y deberían ser fechas o número?
3. ¿Qué otras columnas tienen menos «non-null» que el total?

In [ ]:
crudo = pd.read_csv(dato("reservas_sucio.csv"))
print(crudo.shape)
crudo.info()

### 👀 A3 · EJEMPLO — Resumen de calidad

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué columna tiene el mayor porcentaje de nulos y cuál es?
2. `id_reserva` tiene un número de valores distintos y la tabla un número de filas: ¿cuáles son y qué indica?
3. ¿Cuántos valores distintos tienen `canal` y `pais_origen`? ¿Te parecen demasiados para unas pocas categorías?

In [ ]:
def resumen_calidad(tabla):
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "nulos": tabla.isna().sum(),
        "pct_nulos": (tabla.isna().mean() * 100).round(2),
        "distintos": tabla.nunique(),
    })

resumen_calidad(crudo)

### 👀 A4 · EJEMPLO — Contar duplicados

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Cuántas filas duplicadas exactas hay?
2. Mira una pareja de la tabla: ¿en qué se diferencian sus dos filas?

In [ ]:
print("Filas duplicadas exactas:", crudo.duplicated().sum())
crudo[crudo.duplicated(keep=False)].sort_values("id_reserva").head(6)

### ✏️ A5 · EJERCICIO 1 de 5 — Quitar duplicados

**Tu turno:** Guarda en `n_duplicados` cuántas filas están repetidas y crea `d1`, una copia de `crudo` **sin duplicados** y con el índice reiniciado.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen `n_duplicados` y `len(d1)`?
2. ¿Por qué `len(d1)` coincide con el número de `id_reserva` distintos del resumen de calidad?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: duplicated().sum() y drop_duplicates().reset_index(drop=True)
n_duplicados = ...
d1 = ...

assert n_duplicados == 45, "Cuenta las filas duplicadas con crudo.duplicated().sum()"
assert len(d1) == 1500, "Tras quitar duplicados deben quedar 1500 reservas"
assert d1["id_reserva"].is_unique, "id_reserva debería ser único tras deduplicar"
print("✔ Ejercicio 1 correcto")

### 👀 A6 · EJEMPLO — Precios con coma decimal

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos precios usan coma decimal?
2. ¿Por qué pandas leyó toda la columna `precio_noche` como texto?

In [ ]:
con_coma = d1["precio_noche"].astype(str).str.contains(",").sum()
print("Precios con coma decimal:", con_coma)

### ✏️ A7 · EJERCICIO 2 de 5 — Precio como número

**Tu turno:** Crea `d2` (copia de `d1`) donde `precio_noche` sea numérico (float), cambiando la coma por punto. Guarda en `nulos_precio` el número de nulos resultantes.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuánto vale `nulos_precio` y de dónde salen?
2. ¿Qué tipo tiene ahora `precio_noche`?
3. ¿Qué pasaría si no cambiaras la coma por punto antes de convertir?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: d1.copy(); .astype(str).str.replace(",", ".") y pd.to_numeric(..., errors="coerce")
d2 = ...
nulos_precio = ...

assert d2["precio_noche"].dtype == float, "precio_noche debe ser float"
assert 60 <= nulos_precio <= 90, "Los nulos de precio deberían ser los ya existentes (unos 5 %); si salen muchos más, revisa la conversión"
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Leer los dos formatos de fecha

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuántas fechas se leen con cada formato?
2. ¿Cuántas quedan sin leer y qué significa que 1350 + 150 sume 1500?

In [ ]:
iso = pd.to_datetime(d2["fecha_reserva"], format="%Y-%m-%d", errors="coerce")
espanol = pd.to_datetime(d2["fecha_reserva"], format="%d/%m/%Y", errors="coerce")
print("ISO:", iso.notna().sum(), "· dd/mm/aaaa:", espanol.notna().sum(), "· sin leer:", (iso.isna() & espanol.isna()).sum())

### ✏️ A9 · EJERCICIO 3 de 5 — Fechas como fechas

**Tu turno:** Crea `d3` (copia de `d2`) con `fecha_reserva` y `fecha_llegada` como fechas (`datetime64`). Para `fecha_reserva` combina los dos formatos (pista: `iso.fillna(espanol)`).

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Entre qué fechas están las reservas (`fecha_reserva`)?
2. ¿Qué pasa si lees las 150 fechas con barras como mes/día (`%m/%d/%Y`) en vez de día/mes/año?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: iso.fillna(espanol) para fecha_reserva; pd.to_datetime para fecha_llegada
d3 = ...

assert str(d3["fecha_reserva"].dtype).startswith("datetime"), "fecha_reserva debe ser datetime"
assert d3["fecha_reserva"].isna().sum() == 0, "No debe quedar ninguna fecha sin leer"
assert d3["fecha_reserva"].min().year == 2024, "Comprueba que no has intercambiado día y mes"
print("✔ Ejercicio 3 correcto")

### 👀 A10 · EJEMPLO — Ver las categorías

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿De cuántas formas distintas está escrito el canal `web_intermediaria`?
2. En `pais_origen`, ¿qué valores son en realidad el mismo?

In [ ]:
print(d3["canal"].value_counts().to_string())
print()
print(d3["pais_origen"].value_counts().to_string())

### ✏️ A11 · EJERCICIO 4 de 5 — Normalizar canal y país

**Tu turno:** Normaliza `canal`: sin espacios sobrantes, en minúsculas y con `_` en lugar de espacios. Después normaliza `pais_origen` con el diccionario `equivalencias` para que queden solo los 8 valores válidos (`ES, UK, DE, NL, IT, FR, Nordicos, otros`). Guarda el resultado en `d4`.

**Debe salir:** ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Cuántos canales y cuántos países quedan?
2. ¿Qué hace `.map(equivalencias)` con un valor que no está en el diccionario y cómo se detecta?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A11, **después de intentarlo**.

In [ ]:
equivalencias = {"es": "ES", "españa": "ES", "uk": "UK", "reino unido": "UK", "de": "DE",
                 "nl": "NL", "it": "IT", "fr": "FR", "nordicos": "Nordicos", "otros": "otros"}
# pista: .str.strip().str.lower().str.replace(" ", "_") y .str.strip().str.lower().map(equivalencias)
d4 = ...

assert sorted(d4["canal"].unique()) == ["agencia", "directo", "empresa", "web_intermediaria"], "Canal: deben quedar 4 valores"
assert d4["pais_origen"].nunique() == 8, "País: deben quedar 8 valores (los nulos no cuentan)"
assert d4["pais_origen"].isna().sum() == d3["pais_origen"].isna().sum(), "El mapeo no debe crear nulos nuevos: revisa el diccionario"
print("✔ Ejercicio 4 correcto")

### 👀 A12 · EJEMPLO — Estadísticos y diagramas de caja

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué mínimos o máximos son imposibles?
2. ¿Por qué la media del precio (130,2) queda tan lejos de la mediana (93,2)?
3. En el diagrama de la derecha, sin los imposibles, ¿qué son los puntos sueltos?

In [ ]:
print(d4[["precio_noche", "noches", "antelacion_dias"]].describe().round(1))
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x=d4["precio_noche"], ax=ejes[0], color="#8ecae6")
ejes[0].set(title="Precio con outliers", xlabel="Precio por noche (€)")
sns.boxplot(x=d4.loc[d4["precio_noche"].between(1, 1000), "precio_noche"], ax=ejes[1], color="#8ecae6")
ejes[1].set(title="Precio sin los valores imposibles", xlabel="Precio por noche (€)")
plt.tight_layout()
plt.show()

### ✏️ A13 · EJERCICIO 5 de 5 — Valores imposibles a nulo

**Tu turno:** Crea `d5` (copia de `d4`) aplicando estas reglas de negocio, convirtiendo en `np.nan` (no borrando filas) los valores que las incumplan: - `precio_noche` debe estar entre 20 y 1000 €, - `noches` debe estar entre 1 y 60. (La antelación negativa la trataremos en la sección 7, cruzándola con las fechas.) Guarda en `n_imposibles` cuántos valores has convertido en nulo en total.

**Debe salir:** ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Cuánto vale `n_imposibles` y de qué reglas sale?
2. ¿Por qué se resta `antes`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A13, **después de intentarlo**.

In [ ]:
# pista: .where(condición) conserva lo que cumple y pone nulo en el resto; .between(a, b) dice si está en el rango
d5 = ...
n_imposibles = ...

assert d5["precio_noche"].dropna().between(20, 1000).all(), "Quedan precios fuera de rango"
assert (d5["noches"].dropna() >= 1).all(), "Quedan noches <= 0"
assert 8 <= n_imposibles <= 14, "Deberían ser unos 10-14 valores imposibles (precios y noches). Revisa las reglas"
print("✔ Ejercicio 5 correcto")

### 👀 A14 · EJEMPLO — Recalcular la antelación

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A14).

1. ¿Cuántas antelaciones eran negativas y cuántas no cuadraban con las fechas?
2. ¿Dónde estaba el error, en la columna `antelacion_dias` o en las fechas?
3. Ejecuta la celda otra vez: ¿qué sale ahora?

In [ ]:
antelacion_calculada = (d5["fecha_llegada"] - d5["fecha_reserva"]).dt.days
incoherentes = antelacion_calculada != d5["antelacion_dias"]
print("Antelaciones negativas:", (d5["antelacion_dias"] < 0).sum())
print("Antelaciones que no cuadran con las fechas:", incoherentes.sum())
# Las fechas mandan: recalculamos toda la columna desde ellas
d5["antelacion_dias"] = antelacion_calculada
print("Mínimo tras recalcular:", d5["antelacion_dias"].min())

### 👀 A15 · EJEMPLO — Nulos que quedan

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A15).

1. ¿Qué columna tiene más nulos ahora y qué porcentaje?
2. `precio_noche` tiene 87 nulos: ¿de dónde salen?
3. ¿Hemos rellenado algún nulo?

In [ ]:
resumen_calidad(d5).query("nulos > 0")

### ✏️ A16 · MINI-RETO — Tu función limpiar_reservas

**Tu turno:** Junta todo lo anterior en una función que reciba el DataFrame crudo y devuelva **dos cosas**: la tabla limpia y un diccionario `resumen` que cuente lo que has cambiado (`filas_antes`, `duplicados_quitados`, `imposibles_a_nulo`, `antelaciones_recalculadas`, `filas_despues`). Debe ser **reproducible** (misma entrada → misma salida) y **no modificar** el original. Pasos: duplicados → precio a número → fechas → categorías → imposibles a nulo → antelación recalculada. Llámala así: `limpio, resumen = limpiar_reservas(crudo)`.

**Debe salir:** ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A16).

1. ¿Qué valor tiene `resumen`?
2. ¿Por qué `crudo` sigue teniendo 1545 filas después de llamar a la función?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A16, **después de intentarlo**.

In [ ]:
# pista: reutiliza el código de los ejercicios 1 a 5 y de la sección 7, en este orden:
#        duplicados → precio → fechas → categorías → imposibles → antelación
def limpiar_reservas(tabla):
    t = tabla.copy()
    resumen = {"filas_antes": len(t)}
    ...
    return t, resumen

limpio, resumen = limpiar_reservas(crudo)

assert len(crudo) == 1545, "La función no debe modificar el DataFrame original"
assert limpio.shape[0] == 1500, "Deben quedar 1500 filas"
assert limpio["id_reserva"].is_unique
assert limpio["canal"].nunique() == 4 and limpio["pais_origen"].nunique() == 8
assert limpio["precio_noche"].dtype == float
assert (limpio["antelacion_dias"] >= 0).all() and (limpio["noches"].dropna() > 0).all()
assert str(limpio["fecha_reserva"].dtype).startswith("datetime")
assert resumen["filas_antes"] == 1545 and resumen["duplicados_quitados"] == 45, "El resumen debe contar las filas de partida y los duplicados quitados"
assert resumen["filas_despues"] == limpio.shape[0]
print("✔ Mini-reto correcto")

### 👀 A17 · EJEMPLO — Comparación antes y después

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A17).

1. ¿Cuánto cambian filas, duplicados, canales y países entre «antes» y «después»?

In [ ]:
# Comparación antes/después
comparacion = pd.DataFrame({
    "antes": [len(crudo), crudo.duplicated().sum(), crudo["canal"].nunique(), crudo["pais_origen"].nunique()],
    "después": [len(limpio), limpio.duplicated().sum(), limpio["canal"].nunique(), limpio["pais_origen"].nunique()],
}, index=["filas", "duplicados", "valores distintos de canal", "valores distintos de país"])
comparacion

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `pandas` (tablas), `numpy` (números), `matplotlib` y `seaborn` (gráficos).
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - `SEMILLA = 42` fija el azar para que todos obtengáis lo mismo.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Diagnóstico inicial

#### A2 · Ejemplo — Cargar el fichero sucio

Nunca limpies a ciegas: primero mide. Guardamos siempre el original (`crudo`) y trabajamos sobre una copia, para poder comparar antes y después.

- **Qué hace:**
    - `dato("reservas_sucio.csv")` devuelve dónde está el archivo y `pd.read_csv(...)` lo lee como tabla (*DataFrame*) llamada `crudo`.
    - Aquí **no** se piden las fechas como fechas (`parse_dates`): el fichero las trae mezcladas y hay que arreglarlas a mano más adelante.
    - `crudo.shape` escribe `(filas, columnas)` y `crudo.info()` lista cada columna con su **tipo** y cuántos valores **no nulos** tiene.
- **Qué debe salir:**
    - `(1545, 18)` y la lista de 18 columnas.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas filas y cuántas columnas tiene `crudo`?* → **1545 filas y 18 columnas.**
2. *¿Qué columnas están como texto (`object`) y deberían ser fechas o número?* → `fecha_reserva` y `fecha_llegada` (fechas) y `precio_noche` (número). Según la versión de pandas puede poner `str` en vez de `object`. Además `adultos` es decimal (`float64`) solo porque tiene nulos: 1499 no nulos de 1545.
3. *¿Qué otras columnas tienen menos «non-null» que el total?* → `regimen` (1484), `pais_origen` (1430) y `precio_noche` (1461): tienen huecos.

</details>

#### A3 · Ejemplo — Resumen de calidad

- **Qué hace:**
    - `def resumen_calidad(tabla):` crea una **función**: un trozo de código con nombre que se puede usar con cualquier tabla.
    - Dentro monta una tabla con, por columna: el tipo, cuántos nulos tiene, el % de nulos y cuántos valores distintos.
    - `resumen_calidad(crudo)` la usa con nuestra tabla. La función queda guardada para usarla más veces.
- **Qué debe salir:**
    - Una tabla de 18 filas (una por columna) con `tipo`, `nulos`, `pct_nulos` y `distintos`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué columna tiene el mayor porcentaje de nulos y cuál es?* → `pais_origen`: 115 nulos, el 7,4 %. Le siguen `precio_noche` (5,4 %), `regimen` (4,0 %) y `adultos` (3,0 %). El resto está completo.
2. *`id_reserva` tiene un número de valores distintos y la tabla un número de filas: ¿cuáles son y qué indica?* → 1500 distintos y 1545 filas: **hay 45 reservas repetidas**.
3. *¿Cuántos valores distintos tienen `canal` y `pais_origen`? ¿Te parecen demasiados para unas pocas categorías?* → 9 y 12: demasiados. Hay solo 4 canales y 8 países reales, así que hay categorías escritas de más de una forma (lo veremos en la sección 5).

</details>

## 2. Duplicados

#### A4 · Ejemplo — Contar duplicados

Un duplicado exacto suele venir de una exportación repetida o de un doble envío. Si no se quita, esa reserva «pesa» dos veces y, peor, puede acabar en entrenamiento **y** en test.

- **Qué hace:**
    - `crudo.duplicated()` marca con verdadero cada fila que ya apareció antes; `.sum()` cuenta los verdaderos.
    - `crudo[crudo.duplicated(keep=False)]` enseña **todas** las filas repetidas (con `keep=False` salen la original y su copia); `.sort_values("id_reserva")` las ordena para ver las parejas juntas y `.head(6)` muestra 6.
- **Qué debe salir:**
    - `Filas duplicadas exactas: 45` y una tabla de 6 filas, en parejas (reservas 192, 359 y 511).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas filas duplicadas exactas hay?* → **45.**
2. *Mira una pareja de la tabla: ¿en qué se diferencian sus dos filas?* → En nada: son idénticas en todas las columnas. Es la misma reserva exportada dos veces (las parejas son las reservas 192, 359 y 511).

</details>

#### A5 · Ejercicio 1 — Quitar duplicados

- **Qué hace:**
    - `crudo.duplicated().sum()` cuenta las filas repetidas (como en el ejemplo).
    - `drop_duplicates()` quita las copias y se queda con la primera de cada pareja.
    - `reset_index(drop=True)` vuelve a numerar las filas de 0 en adelante y descarta la numeración vieja.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `n_duplicados` vale 45 y `d1` tiene 1500 filas.
- **Si falla:**
    - `Cuenta las filas duplicadas con crudo.duplicated().sum()` → `n_duplicados` no vale 45.
    - `Tras quitar duplicados deben quedar 1500 reservas` → `d1` no es la tabla sin repetidas (¿falta asignar el resultado a `d1`?).
    - `id_reserva debería ser único tras deduplicar` → siguen quedando copias.
    - `NameError: name 'crudo' is not defined` → no has ejecutado el primer ejemplo de la sección 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `n_duplicados` y `len(d1)`?* → **45 y 1500.**
2. *¿Por qué `len(d1)` coincide con el número de `id_reserva` distintos del resumen de calidad?* → Porque 1545 − 45 = 1500: al quitar las copias queda **una fila por reserva**, y `id_reserva` ya es único.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
n_duplicados = int(crudo.duplicated().sum())
d1 = crudo.drop_duplicates().reset_index(drop=True)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Tipos incorrectos: números escritos como texto

#### A6 · Ejemplo — Precios con coma decimal

Algunos precios llevan **coma decimal** («84,36»), típica de una exportación de Excel en español. Por eso toda la columna es texto. Se corrige reemplazando la coma y convirtiendo con `pd.to_numeric`. Con `errors="coerce"` lo que no se pueda convertir pasa a nulo (mejor que romper el proceso, pero **hay que contarlo**).

- **Qué hace:**
    - `d1["precio_noche"].astype(str)` pasa todos los precios a texto.
    - `.str.contains(",")` marca con verdadero los que llevan coma y `.sum()` los cuenta.
- **Qué debe salir:**
    - `Precios con coma decimal: 147`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos precios usan coma decimal?* → **147.**
2. *¿Por qué pandas leyó toda la columna `precio_noche` como texto?* → Porque la columna mezcla precios con coma y con punto: basta un valor que no sea un número válido en inglés para que pandas la deje entera como texto.

</details>

#### A7 · Ejercicio 2 — Precio como número

- **Qué hace:**
    - `d1.copy()` hace una copia, para no tocar `d1`.
    - `.astype(str).str.replace(",", ".")` cambia la coma por punto en el texto.
    - `pd.to_numeric(..., errors="coerce")` lo convierte a número decimal; lo que no pueda convertir queda como nulo (`NaN`).
    - `.isna().sum()` cuenta los nulos de `precio_noche`.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. `nulos_precio` vale 79 y `precio_noche` es de tipo `float64`.
- **Si falla:**
    - `precio_noche debe ser float` → falta `pd.to_numeric(...)`.
    - `Los nulos de precio deberían ser los ya existentes...` → han aparecido nulos nuevos: revisa que cambias la coma por punto antes de convertir.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `nulos_precio` y de dónde salen?* → **79**: son los que **ya había** en `d1`. La conversión no ha roto ningún precio.
2. *¿Qué tipo tiene ahora `precio_noche`?* → `float64` (número decimal).
3. *¿Qué pasaría si no cambiaras la coma por punto antes de convertir?* → `pd.to_numeric` no entiende «84,36» y, con `errors="coerce"`, lo pasaría a nulo **sin avisar**: los nulos subirían de 79 a 226 (los 147 precios con coma, perdidos). Por eso se cuenta siempre.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
d2 = d1.copy()
d2["precio_noche"] = pd.to_numeric(d2["precio_noche"].astype(str).str.replace(",", "."), errors="coerce")
nulos_precio = int(d2["precio_noche"].isna().sum())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Fechas en formatos mezclados

#### A8 · Ejemplo — Leer los dos formatos de fecha

`fecha_reserva` mezcla `2024-11-19` (ISO) con `19/11/2024` (día/mes/año). Ojo con la trampa: `05/11/2024` podría leerse como 5 de noviembre o 11 de mayo. **No adivines**: como sabemos que el formato «con barras» es día/mes/año, se lee cada grupo con su formato y se combinan.

- **Qué hace:**
    - `pd.to_datetime(..., format="%Y-%m-%d", errors="coerce")` lee como fecha solo las que tienen forma año-mes-día; las demás quedan como `NaT` (fecha nula). `%Y` es el año de 4 cifras, `%m` el mes y `%d` el día.
    - La segunda línea hace lo mismo con `"%d/%m/%Y"` (día/mes/año).
    - `iso.notna().sum()` cuenta las leídas con cada formato y `(iso.isna() & espanol.isna()).sum()` cuenta las que no ha leído ninguno.
- **Qué debe salir:**
    - `ISO: 1350 · dd/mm/aaaa: 150 · sin leer: 0`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas fechas se leen con cada formato?* → **1350** con el formato ISO (año-mes-día) y **150** con día/mes/año.
2. *¿Cuántas quedan sin leer y qué significa que 1350 + 150 sume 1500?* → **Ninguna** queda sin leer. Todas se leen con uno de los dos formatos y ninguna con los dos, así que no hay ambigüedad.

</details>

#### A9 · Ejercicio 3 — Fechas como fechas

- **Qué hace:**
    - `iso.fillna(espanol)` rellena los huecos (`NaT`) de `iso` con lo que leyó `espanol`: queda una sola serie con todas las fechas.
    - `fecha_llegada` trae un único formato, así que basta `pd.to_datetime(..., format="%Y-%m-%d")`.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. Las dos fechas son de tipo `datetime64` y no queda ninguna sin leer.
- **Si falla:**
    - `fecha_reserva debe ser datetime` → falta convertirla (`iso.fillna(espanol)`).
    - `No debe quedar ninguna fecha sin leer` → has usado solo un formato.
    - `Comprueba que no has intercambiado día y mes` → la primera fecha de reserva no sale en 2024: revisa los formatos (`%Y-%m-%d` y `%d/%m/%Y`).
    - `NameError: name 'iso' is not defined` → no has ejecutado el ejemplo de esta sección.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué fechas están las reservas (`fecha_reserva`)?* → Del **2024-01-01** al **2025-12-30**.
2. *¿Qué pasa si lees las 150 fechas con barras como mes/día (`%m/%d/%Y`) en vez de día/mes/año?* → **94 fallan** (no existe el mes 19, 27…) y quedan sin leer; las otras **56 se leen mal sin avisar** (el 05/11 pasa a ser el 11 de mayo). Por eso no se adivina el formato: se lee cada grupo con el suyo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
d3 = d2.copy()
d3["fecha_reserva"] = iso.fillna(espanol)
d3["fecha_llegada"] = pd.to_datetime(d3["fecha_llegada"], format="%Y-%m-%d")
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Categorías inconsistentes

#### A10 · Ejemplo — Ver las categorías

«Directo», «directo » y «DIRECTO» son lo mismo para una persona, pero tres categorías distintas para el ordenador. Se normaliza: quitar espacios, pasar a minúsculas, unificar separadores y, si hace falta, un diccionario de equivalencias.

- **Qué hace:**
    - `value_counts()` cuenta cuántas veces sale cada valor; `.to_string()` lo escribe entero, sin recortar.
    - `print()` suelto deja una línea en blanco entre las dos listas.
- **Qué debe salir:**
    - Dos listas: `canal` con 9 valores y `pais_origen` con 12.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿De cuántas formas distintas está escrito el canal `web_intermediaria`?* → **De 4**: `web_intermediaria`, `WEB_INTERMEDIARIA`, `Web Intermediaria` y `web intermediaria`. Y `directo` de 3 (una lleva un espacio al final que no se ve): son 9 canales donde debería haber 4.
2. *En `pais_origen`, ¿qué valores son en realidad el mismo?* → `ES`, `España` y `es` por un lado, y `UK`, `Reino Unido` y `uk` por otro: 12 valores donde debería haber 8.

</details>

#### A11 · Ejercicio 4 — Normalizar canal y país

- **Qué hace:**
    - `equivalencias` es un diccionario: cada variante en minúsculas (`"españa"`, `"reino unido"`…) apunta a su forma buena (`"ES"`, `"UK"`…).
    - `.str.strip()` quita espacios de los extremos, `.str.lower()` pasa a minúsculas y `.str.replace(" ", "_")` cambia los espacios por `_`.
    - `.map(equivalencias)` sustituye cada valor por su equivalente; lo que no esté en el diccionario pasa a nulo.
- **Qué debe salir:**
    - `✔ Ejercicio 4 correcto`. `canal` queda con 4 valores (`agencia`, `directo`, `empresa`, `web_intermediaria`) y `pais_origen` con 8.
- **Si falla:**
    - `Canal: deben quedar 4 valores` → falta `strip`, `lower` o el cambio de espacios por `_`.
    - `País: deben quedar 8 valores` → falta `.map(equivalencias)` o `lower()` antes del `map`.
    - `El mapeo no debe crear nulos nuevos` → alguna variante no está en el diccionario o no has quitado los espacios antes de buscarla.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos canales y cuántos países quedan?* → **4** canales (`agencia`, `directo`, `empresa`, `web_intermediaria`) y **8** países (`ES`, `UK`, `DE`, `NL`, `IT`, `FR`, `Nordicos`, `otros`).
2. *¿Qué hace `.map(equivalencias)` con un valor que no está en el diccionario y cómo se detecta?* → Lo deja **nulo**. Se detecta comprobando que los nulos de `pais_origen` no aumentan: eran 112 antes y siguen siendo 112.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
equivalencias = {"es": "ES", "españa": "ES", "uk": "UK", "reino unido": "UK", "de": "DE",
                 "nl": "NL", "it": "IT", "fr": "FR", "nordicos": "Nordicos", "otros": "otros"}
d4 = d3.copy()
d4["canal"] = d4["canal"].str.strip().str.lower().str.replace(" ", "_")
d4["pais_origen"] = d4["pais_origen"].str.strip().str.lower().map(equivalencias)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Valores imposibles y *outliers*

#### A12 · Ejemplo — Estadísticos y diagramas de caja

Hay dos clases de valores extraños. **Imposibles** (regla de negocio rota: precio ≤ 0, noches ≤ 0, antelación negativa): son **errores**, se convierten en nulo (y se imputarán después) o se corrigen si se puede. **Raros pero posibles** (una suite de 600 €): son *outliers* y **no se borran sin motivo**. Una herramienta rápida para detectarlos es la regla del **rango intercuartílico (IQR)**: sospechoso lo que queda fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`.

- **Qué hace:**
    - `.describe()` resume tres columnas: cuántos datos hay, media, desviación, mínimo, cuartiles y máximo; `.round(1)` deja un decimal.
    - `plt.subplots(1, 2, ...)` prepara dos huecos de gráfico, uno al lado del otro.
    - `sns.boxplot` dibuja un **diagrama de caja**: la caja es el 50 % central, la raya es la mediana y los puntos sueltos son valores atípicos. El de la izquierda usa todos los precios y el de la derecha solo los de entre 1 y 1000 €.
- **Qué debe salir:**
    - Una tabla con el resumen de `precio_noche`, `noches` y `antelacion_dias`, y dos diagramas de caja.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué mínimos o máximos son imposibles?* → `precio_noche` va de −50 a 9999 € por noche (ni un precio negativo ni uno de 9999 € son posibles), `noches` tiene un mínimo de −2 y `antelacion_dias` de −3.
2. *¿Por qué la media del precio (130,2) queda tan lejos de la mediana (93,2)?* → Porque los valores extremos (el 9999 y el −50) arrastran la media. Con datos así conviene mirar la **mediana**.
3. *En el diagrama de la derecha, sin los imposibles, ¿qué son los puntos sueltos?* → *Outliers*: precios altos raros pero posibles (una suite cara). **No se borran sin motivo.** En el diagrama de la izquierda el 9999 aplasta todo lo demás.

</details>

#### A13 · Ejercicio 5 — Valores imposibles a nulo

- **Qué hace:**
    - `d4.copy()` hace una copia y `antes` guarda cuántos nulos había ya en precio y noches.
    - `.between(20, 1000)` da verdadero si el valor está en el rango; `.where(...)` conserva el valor si se cumple y pone `NaN` si no. **No se borran filas.**
    - `n_imposibles` es el número de nulos de después menos los de antes: solo cuentan los nuevos.
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto`. `n_imposibles` vale 14.
- **Si falla:**
    - `Quedan precios fuera de rango` → falta aplicar el rango a `precio_noche`.
    - `Quedan noches <= 0` → falta aplicar el rango a `noches`.
    - `Deberían ser unos 10-14 valores imposibles...` → has contado también los nulos que ya había (o las reglas están mal escritas).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `n_imposibles` y de qué reglas sale?* → **14**: 4 precios ≤ 0, 4 precios > 1000 y 6 noches ≤ 0.
2. *¿Por qué se resta `antes`?* → Porque ya había nulos de precio (79) que no son imposibles, sino datos que faltaban: solo interesan los **nuevos**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
d5 = d4.copy()
antes = d5[["precio_noche", "noches"]].isna().sum().sum()
d5["precio_noche"] = d5["precio_noche"].where(d5["precio_noche"].between(20, 1000))
d5["noches"] = d5["noches"].where(d5["noches"].between(1, 60))
n_imposibles = int(d5[["precio_noche", "noches"]].isna().sum().sum() - antes)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 7. Consistencia entre columnas

#### A14 · Ejemplo — Recalcular la antelación

Además de mirar cada columna, hay que cruzarlas. Aquí `antelacion_dias` debe ser igual a `fecha_llegada − fecha_reserva` en días. Si no cuadra, la fuente de verdad son **las fechas** (más difíciles de falsear por error).

- **Qué hace:**
    - Restar dos fechas da una duración; `.dt.days` la pasa a número de días.
    - `incoherentes` marca las filas donde ese cálculo no coincide con la columna `antelacion_dias`.
    - Las dos primeras líneas con `print` cuentan las negativas y las que no cuadran.
    - `d5["antelacion_dias"] = antelacion_calculada` **sobrescribe** la columna con el valor calculado desde las fechas.
- **Qué debe salir:**
    - `Antelaciones negativas: 5`, `Antelaciones que no cuadran con las fechas: 5` y `Mínimo tras recalcular: 1`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas antelaciones eran negativas y cuántas no cuadraban con las fechas?* → **5 y 5**: las negativas son justo las que no cuadraban.
2. *¿Dónde estaba el error, en la columna `antelacion_dias` o en las fechas?* → En la columna: por eso se recalcula desde las fechas (las fechas mandan). Tras recalcular, el mínimo es 1 día.
3. *Ejecuta la celda otra vez: ¿qué sale ahora?* → `0` negativas y `0` que no cuadran: la columna ya está corregida.

</details>

## 8. Nulos: ¿qué hacemos con ellos?

#### A15 · Ejemplo — Nulos que quedan

Antes de imputar, pregunta **por qué falta** el dato. Si falta al azar y son pocos: imputar (mediana en numéricas, moda o «desconocido» en categóricas). Si la ausencia tiene significado (p. ej., país no informado en reservas por teléfono): dejar una categoría `"desconocido"` o una columna indicadora. Si falta casi todo en una columna: valorar quitarla. **Regla de oro:** los métodos que *aprenden* de los datos (mediana, moda, escalado) se ajustan **solo con el conjunto de entrenamiento**, dentro de un *pipeline*. Lo veremos en la práctica 3. Aquí solo contamos y documentamos.

- **Qué hace:**
    - `resumen_calidad(d5)` usa la función de la sección 1 con la tabla ya limpiada.
    - `.query("nulos > 0")` se queda solo con las columnas que tienen algún nulo.
- **Qué debe salir:**
    - Una tabla de 5 filas: `noches` 6, `adultos` 46, `regimen` 59, `pais_origen` 112 y `precio_noche` 87.
- **Si falla:**
    - `NameError: name 'resumen_calidad' is not defined` → no has ejecutado el ejemplo «Resumen de calidad» de la sección 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué columna tiene más nulos ahora y qué porcentaje?* → `pais_origen`: 112 nulos, el 7,5 %.
2. *`precio_noche` tiene 87 nulos: ¿de dónde salen?* → 79 que ya faltaban más 8 precios imposibles que pasamos a nulo. `noches` tiene 6: los 6 imposibles.
3. *¿Hemos rellenado algún nulo?* → No: aquí solo contamos y documentamos. Rellenarlos se hace después, dentro del *pipeline*.

</details>

## 9. Mini-reto: tu función `limpiar_reservas`

#### A16 · Mini-reto — Tu función limpiar_reservas

- **Qué hace:**
    - Una **función** que junta todos los pasos de limpieza: recibe la tabla cruda y devuelve dos cosas, la tabla limpia y un diccionario `resumen`.
    - `tabla.copy()` evita modificar el original. El `resumen` va contando lo que cambia en cada paso.
    - El orden importa: duplicados → precio → fechas → categorías → imposibles → antelación.
- **Qué debe salir:**
    - `✔ Mini-reto correcto`. `resumen` vale `filas_antes` 1545, `duplicados_quitados` 45, `imposibles_a_nulo` 14, `antelaciones_recalculadas` 5 y `filas_despues` 1500.
- **Si falla:**
    - `La función no debe modificar el DataFrame original` → falta `tabla.copy()` o alguna operación escribe sobre `tabla`.
    - `Deben quedar 1500 filas` o `id_reserva` no único → falta quitar duplicados.
    - `El resumen debe contar las filas de partida y los duplicados quitados` → faltan las claves `filas_antes` o `duplicados_quitados`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valor tiene `resumen`?* → `filas_antes` 1545, `duplicados_quitados` 45, `imposibles_a_nulo` 14, `antelaciones_recalculadas` 5 y `filas_despues` 1500.
2. *¿Por qué `crudo` sigue teniendo 1545 filas después de llamar a la función?* → Porque la función trabaja sobre `tabla.copy()`: no modifica el original. Y es reproducible: misma entrada, misma salida.

</details>

#### A17 · Ejemplo — Comparación antes y después

- **Qué hace:**
    - Monta una tabla con cuatro medidas de `crudo` («antes») y de `limpio` («después»): filas, duplicados y valores distintos de `canal` y de `pais_origen`.
- **Qué debe salir:**
    - Una tabla: filas 1545 → 1500, duplicados 45 → 0, canales distintos 9 → 4 y países distintos 12 → 8.
- **Si falla:**
    - `NameError: name 'limpio' is not defined` → falta ejecutar antes el mini-reto.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto cambian filas, duplicados, canales y países entre «antes» y «después»?* → Filas 1545 → 1500, duplicados 45 → 0, canales distintos 9 → 4 y países distintos 12 → 8. Así se **documenta** el efecto de la limpieza.

</details>

## 10. Para reflexionar

1. ¿Por qué convertimos los imposibles en nulo en lugar de borrar la fila entera?
2. Un precio de 600 € en una suite en agosto, ¿es un error? ¿Qué harías con él?
3. Si el 40 % de una columna estuviera vacío, ¿qué opciones hay y qué riesgos tiene cada una?
4. ¿Qué pregunta harías **al cliente** para saber por qué hay precios con coma y fechas mezcladas?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque borrar la fila entera tira también los demás datos válidos de esa reserva (otras 16 columnas). Con un nulo se conserva la fila, se puede imputar después y queda contado cuánto se ha tocado.
2. No tiene por qué: una suite en agosto puede costar 600 €. Es un valor **raro pero posible**: se investiga (se compara con su tipo de habitación y su fecha, y se pregunta al cliente) y no se borra sin motivo.
3. Quitar la columna (se pierde su información), imputar con mediana o moda (se inventa dato y se reduce la variabilidad) o dejar una categoría «desconocido» o una columna indicadora (si la ausencia significa algo). Con tanto vacío cualquier imputación es arriesgada, sobre todo si no falta al azar.
4. Por ejemplo: ¿de qué sistemas se exporta cada columna y con qué configuración regional (coma decimal, formato de fecha)? ¿Cómo se registra el país cuando el cliente no lo informa?

</details>

## 11. Resumen: qué has aprendido
- Diagnostica antes de limpiar y conserva siempre el original.
- Defectos típicos: duplicados, tipos, formatos de fecha, categorías, imposibles, nulos e incoherencias entre columnas.
- Imposible ≠ raro: los errores se corrigen; los raros se investigan.
- La imputación y el escalado se harán **dentro del pipeline** para evitar fugas.